In [ ]:
import time
import torch
import numpy as np
import gymnasium as gym

from modules.agent_nets.modular import ModularAgentNetwork
from modules.backbones.mlp import MLPBackbone
from modules.heads.policy import PolicyHeads
from modules.heads.value import ValueHead
from agents.learner.losses.representations import (
    ClassificationRepresentation,
    ScalarRepresentation,
)
from agents.action_selectors.selectors import CategoricalSelector
from agents.action_selectors.policy_sources import NetworkPolicySource
from replay_buffers.buffer_factories import create_ppo_buffer
from agents.learner.batch_iterators import PPOEpochIterator
from agents.learner.base import UniversalLearner
from agents.registries.ppo import build_ppo_loss_pipeline

/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


ModuleNotFoundError: No module named 'modules.backbones.mlp'

In [ ]:
# --- Explicit Hyperparameters ---
ENV_ID = "CartPole-v1"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# PPO Core Params
STEPS_PER_EPOCH = 512
NUM_MINIBATCHES = 4
TRAIN_POLICY_ITERATIONS = 4

# Algorithm Params
GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_PARAM = 0.2
ENTROPY_COEF = 0.01
VALUE_COEF = 0.5
LEARNING_RATE = 2.5e-4
TOTAL_STEPS = 256000

# --- Setup Environment & Dimensions ---
env = gym.make(ENV_ID)
obs_dim = env.observation_space.shape
num_actions = env.action_space.n

print(f"Environment: {ENV_ID} | Obs Dim: {obs_dim} | Num Actions: {num_actions}")

In [ ]:
# --- Instantiate Components Explicitly ---

# 1. Agent Network
agent_network = ModularAgentNetwork(
    input_shape=obs_dim,
    num_actions=num_actions,
    components={
        "policy_head": PolicyHead(
            input_shape=obs_dim,
            representation=ClassificationRepresentation(
                num_classes=num_actions,
            ),
            neck=MLPBackbone(
                input_shape=obs_dim,
                hidden_dims=[64, 64],
            ),
            noisy_sigma=0.0,
        ),
        "value_head": ValueHead(
            input_shape=obs_dim,
            representation=ScalarRepresentation(),
            neck=MLPBackbone(
                input_shape=obs_dim,
                hidden_dims=[64, 64],
            ),
            noisy_sigma=0.0,
        ),
    },
    minibatch_size=STEPS_PER_EPOCH // NUM_MINIBATCHES,
    # Pass your refactored arch kwargs here instead of a config object
    # e.g., hidden_dims=[64, 64]
).to(DEVICE)

# 2. Action Selector & Policy Source
action_selector = CategoricalSelector(exploration=True)
policy_source = NetworkPolicySource(agent_network)

# 3. Replay Buffer
replay_buffer = create_ppo_buffer(
    observation_dimensions=obs_dim,
    max_size=STEPS_PER_EPOCH,
    gamma=GAMMA,
    gae_lambda=GAE_LAMBDA,
    num_actions=num_actions,
    observation_dtype=torch.float32,
)

# 4. Optimizer and Loss Pipeline
optimizer = {"default": torch.optim.Adam(agent_network.parameters(), lr=LEARNING_RATE)}

loss_pipeline = build_ppo_loss_pipeline(
    agent_network=agent_network,
    device=DEVICE,
    clip_param=CLIP_PARAM,
    entropy_coefficient=ENTROPY_COEF,
    critic_coefficient=VALUE_COEF,
    minibatch_size=STEPS_PER_EPOCH,
    num_actions=num_actions,
)

# 5. Learner
learner = UniversalLearner(
    agent_network=agent_network,
    device=DEVICE,
    num_actions=num_actions,
    observation_dimensions=obs_dim,
    observation_dtype=torch.float32,
    loss_pipeline=loss_pipeline,
    optimizer=optimizer,
    lr_scheduler={},
    callbacks=[],
    clipnorm=0.5,
)

print("All components initialized successfully!")

In [ ]:
print("Starting explicit PPO training loop...")
start_time = time.time()

steps_collected = 0
current_episode_score = 0.0
state, info = env.reset()

while steps_collected < TOTAL_STEPS:

    # ==========================================
    # PHASE 1: Trajectory Collection
    # ==========================================
    epoch_steps = 0
    trajectory_start_index = replay_buffer.size

    while epoch_steps < STEPS_PER_EPOCH:
        with torch.no_grad():
            obs_tensor = torch.tensor(
                state, dtype=torch.float32, device=DEVICE
            ).unsqueeze(0)

            # 1. Inference
            result = policy_source.get_inference(obs=obs_tensor, info=info)
            action, metadata = action_selector.select_action(
                result=result,
                info=info,
                exploration=True,
            )

            log_prob = metadata["log_prob"]
            value = metadata["value"]
            action_val = action.item()

            # 2. Step Env
            next_state, reward, terminated, truncated, next_info = env.step(action_val)
            done = terminated or truncated

            # 3. Store Transition
            replay_buffer.store(
                observations=state,
                actions=action_val,
                values=float(value.item() if torch.is_tensor(value) else value),
                old_log_probs=float(
                    log_prob.item() if torch.is_tensor(log_prob) else log_prob
                ),
                rewards=reward,
                dones=done,
                info=info,
            )

            state = next_state
            info = next_info
            current_episode_score += reward
            epoch_steps += 1
            steps_collected += 1

            # 4. Handle Episode End / Bootstrap Value
            if done or epoch_steps == STEPS_PER_EPOCH:
                if terminated:
                    last_value = 0.0
                else:
                    # Bootstrap if truncated or epoch ended mid-episode
                    obs_t = torch.tensor(
                        state, dtype=torch.float32, device=DEVICE
                    ).unsqueeze(0)
                    out = agent_network.obs_inference(obs_t)
                    last_value = out.value.item()

                # Process GAE for the finished trajectory slice
                trajectory_end_index = replay_buffer.size
                trajectory_slice = slice(trajectory_start_index, trajectory_end_index)

                if trajectory_end_index > trajectory_start_index:
                    res = replay_buffer.input_processor.finish_trajectory(
                        replay_buffer.buffers,
                        trajectory_slice,
                        last_value=last_value,
                    )
                    if res:
                        for k, v in res.items():
                            replay_buffer.buffers[k][trajectory_slice] = v

                trajectory_start_index = trajectory_end_index

                if done:
                    print(
                        f"Step {steps_collected} | Episode Score: {current_episode_score}"
                    )
                    state, info = env.reset()
                    current_episode_score = 0.0

    # ==========================================
    # PHASE 2: Optimization (Learning)
    # ==========================================
    iterator = PPOEpochIterator(
        replay_buffer=replay_buffer,
        num_epochs=TRAIN_POLICY_ITERATIONS,
        num_minibatches=NUM_MINIBATCHES,
        device=DEVICE,
    )

    for step_metrics in learner.step(iterator):
        # Optional: Print out step_metrics here to monitor loss
        pass

    # Clear buffer for next epoch
    replay_buffer.clear()

total_time = time.time() - start_time
print(f"Training finished in {total_time:.2f} seconds.")